In [2]:
from pyspark.sql import SparkSession

# create local SparkSession
spark = SparkSession.builder \
    .appName("Module6") \
    .master("local[*]") \
    .getOrCreate()

# check Spark version
print("Spark version:", spark.version)

Spark version: 3.5.0


In [5]:
import os

# folder in container that mounts to your PC
data_dir = "/home/jovyan/data"
os.makedirs(data_dir, exist_ok=True)

# download Yellow Taxi November 2025
os.system("wget -O {}/yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet".format(data_dir))

# download zone lookup
os.system("wget -O {}/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv".format(data_dir))

0

In [6]:
# read parquet file
df = spark.read.parquet(f"{data_dir}/yellow_tripdata_2025-11.parquet")
df.show(5)  # show first 5 rows
print("Number of records:", df.count())

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [7]:
import glob

# 1️⃣ Repartition into 4 partitions
df_repart = df.repartition(4)

# 2️⃣ Save to Parquet (inside volume to keep files on PC)
output_path = f"{data_dir}/yellow_2025_11_repart"
df_repart.write.mode("overwrite").parquet(output_path)

# 3️⃣ Check average Parquet file size
parquet_files = glob.glob(f"{output_path}/*.parquet")
sizes_mb = [os.path.getsize(f)/1024/1024 for f in parquet_files]
avg_size = sum(sizes_mb)/len(sizes_mb)
print("Average Parquet file size (MB):", round(avg_size,1))

Average Parquet file size (MB): 24.4


In [8]:
from pyspark.sql.functions import col, to_date

# Filter only trips from November 15, 2025
df_15 = df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15")

# Count number of trips
count_15 = df_15.count()
print("Number of trips on November 15, 2025:", count_15)

Number of trips on November 15, 2025: 162604


In [10]:
from pyspark.sql.functions import unix_timestamp, max as spark_max
# Calculate trip duration in hours
df_with_duration = df.withColumn(
    "duration_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
)

# Find maximum duration
longest_trip = df_with_duration.select(spark_max("duration_hours")).collect()[0][0]
print("Longest trip duration (hours):", round(longest_trip,1))

Longest trip duration (hours): 90.6


In [11]:
# Load zones
zones = spark.read.csv(f"{data_dir}/taxi_zone_lookup.csv", header=True)

# Join with data by PULocationID
df_with_zones = df.join(zones, df.PULocationID == zones.LocationID, how="left")

# Group by Zone and count trips
least_zone = df_with_zones.groupBy("Zone").count().orderBy("count").first()

print("Least frequent pickup zone:", least_zone["Zone"])

Least frequent pickup zone: Governor's Island/Ellis Island/Liberty Island
